# 01 - Xây dựng bảng trung gian theo ngày (Daily Intermediate Tables)
## Mục tiêu kinh doanh
Chuẩn hóa dữ liệu vận hành e-commerce về cùng hạt thời gian theo ngày để doanh nghiệp có thể dự báo doanh thu, lập kế hoạch tồn kho, ngân sách marketing và năng lực fulfillment.

## Vai trò trong pipeline mô hình
Đây là lớp dữ liệu nền tảng. Nếu bước này sai grain hoặc sai logic tổng hợp, toàn bộ mô hình phía sau sẽ lệch tín hiệu và dẫn đến quyết định kinh doanh sai.

## Đầu vào
- Các bảng sạch dạng parquet trong thư mục dataset/interim/*_base.parquet

## Đầu ra
- dataset/interim/daily/sales_daily.parquet
- dataset/interim/daily/web_traffic_daily.parquet
- dataset/interim/daily/orders_daily.parquet
- dataset/interim/daily/payments_daily.parquet
- dataset/interim/daily/order_items_daily.parquet
- dataset/interim/daily/promotions_daily.parquet
- dataset/interim/daily/returns_daily.parquet
- dataset/interim/daily/reviews_daily.parquet
- dataset/interim/daily/shipments_daily.parquet
- dataset/interim/daily/inventory_monthly_summary.parquet

## Giải thích chuyên môn theo nghiệp vụ e-commerce
1. sales_daily: tạo đường cơ sở GMV/Revenue và COGS theo ngày để đo hiệu quả kinh doanh thuần.
2. web_traffic_daily: gom traffic để liên kết đầu phễu (sessions, visitors) với đầu ra doanh thu.
3. orders_daily: theo dõi sức khỏe chuyển đổi và chất lượng đơn (cancelled, delivered, returned).
4. payments_daily: phản ánh hành vi thanh toán, sức mua thực trả và đặc điểm trả góp.
5. order_items_daily: đo cường độ bán theo sản phẩm, mức độ phụ thuộc khuyến mại và stacked promo.
6. promotions_daily: lượng hóa áp lực khuyến mại theo từng ngày để tách tác động giá/khuyến mãi khỏi nhu cầu tự nhiên.
7. returns_daily + reviews_daily + shipments_daily: bổ sung tín hiệu hậu mãi, trải nghiệm giao hàng, chất lượng sản phẩm; đây là nhóm biến có tác động gián tiếp nhưng bền vững đến doanh thu.
8. inventory_monthly_summary: đưa tín hiệu cung ứng (stockout, fill rate, days of supply) vào mô hình để bắt hiện tượng mất doanh thu do thiếu hàng.

## Giá trị cho doanh nghiệp
- Giúp ban điều hành nhìn cùng lúc 3 trục tăng trưởng: nhu cầu thị trường, năng lực vận hành, và chất lượng trải nghiệm khách hàng.
- Tạo tiền đề để dự báo doanh thu có thể hành động được: dự báo xong có thể quy về quyết định ngân sách, tồn kho, và năng lực vận chuyển.
- Giảm rủi ro dùng dữ liệu rời rạc dẫn đến overfit hoặc ra quyết định cảm tính.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

def find_project_root(start: Path = Path.cwd()) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "dataset" / "raw").exists():
            return p
        if (p / "data" / "raw").exists():
            return p
        if (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root()
BASE_DIR = PROJECT_ROOT / "dataset" / "interim"
OUT_DIR = PROJECT_ROOT / "dataset" / "interim" / "daily"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def read_tbl(name):
    p = BASE_DIR / f"{name}_base.parquet"
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing file: {p}")
    return pd.read_parquet(p)

def to_dt(s):
    return pd.to_datetime(s, errors="coerce")

def clean_daily_table(df, date_col):
    df = df.copy()
    df[date_col] = to_dt(df[date_col])
    df = df.dropna(subset=[date_col]).sort_values(date_col).reset_index(drop=True)
    return df


In [2]:
# 1) sales_daily
sales = clean_daily_table(read_tbl("sales"), "date")
sales_daily = (
    sales.groupby("date", as_index=False)
    .agg(revenue=("revenue", "sum"), cogs=("cogs", "sum"))
    .rename(columns={"date": "Date", "revenue": "Revenue", "cogs": "COGS"})
)
sales_daily.to_parquet(os.path.join(OUT_DIR, "sales_daily.parquet"), index=False)
sales_daily.head()


,Date,Revenue,COGS
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79


In [3]:
# 2) web_traffic_daily
wt = clean_daily_table(read_tbl("web_traffic"), "date")
web_traffic_daily = (
    wt.groupby("date", as_index=False).agg(
        sessions=("sessions", "sum"),
        unique_visitors=("unique_visitors", "sum"),
        page_views=("page_views", "sum"),
        bounce_rate=("bounce_rate", "mean"),
        avg_session_duration_sec=("avg_session_duration_sec", "mean")
    )
    .rename(columns={"date": "Date"})
)
web_traffic_daily.to_parquet(os.path.join(OUT_DIR, "web_traffic_daily.parquet"), index=False)
web_traffic_daily.head()


,Date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec
0,2013-01-01,9760,7253,39093,0.00514,102.9
1,2013-01-02,10456,8151,47611,0.00406,120.5
2,2013-01-03,10076,7458,36963,0.00401,263.6
3,2013-01-04,9973,8063,53078,0.00562,151.8
4,2013-01-05,10223,7882,36790,0.00525,168.6


In [4]:
# 3) orders_daily
orders = clean_daily_table(read_tbl("orders"), "order_date")

orders_daily = orders.groupby("order_date", as_index=False).agg(
    total_orders=("order_id", "nunique"),
    cancelled_orders=("order_status", lambda s: (s == "cancelled").sum()),
    delivered_orders=("order_status", lambda s: (s == "delivered").sum()),
    returned_orders=("order_status", lambda s: (s == "returned").sum())
).rename(columns={"order_date": "Date"})

device_daily = pd.crosstab(orders["order_date"], orders["device_type"]).reset_index()
device_daily.columns = ["Date"] + [f"orders_device_{c}" for c in device_daily.columns[1:]]

source_daily = pd.crosstab(orders["order_date"], orders["order_source"]).reset_index()
source_daily.columns = ["Date"] + [f"orders_source_{c}" for c in source_daily.columns[1:]]

orders_daily = orders_daily.merge(device_daily, on="Date", how="left").merge(source_daily, on="Date", how="left")
orders_daily.to_parquet(os.path.join(OUT_DIR, "orders_daily.parquet"), index=False)
orders_daily.head()


,Date,total_orders,cancelled_orders,delivered_orders,returned_orders,orders_device_desktop,orders_device_mobile,orders_device_tablet,orders_source_direct,orders_source_email_campaign,orders_source_organic_search,orders_source_paid_search,orders_source_referral,orders_source_social_media
0,2012-07-04,162,9,134,11,67,69,26,9,14,52,30,16,41
1,2012-07-05,97,9,81,5,43,38,16,6,11,26,18,9,27
2,2012-07-06,93,11,69,7,42,42,9,10,12,28,23,7,13
3,2012-07-07,73,8,55,6,26,34,13,2,13,20,15,12,11
4,2012-07-08,88,5,67,8,30,45,13,7,11,22,18,2,28


In [5]:
# 4) payments_daily (join orders -> order_date)
payments = read_tbl("payments").copy()
pay_join = payments.merge(orders[["order_id", "order_date"]], on="order_id", how="left")
pay_join["order_date"] = to_dt(pay_join["order_date"])

payments_daily = pay_join.groupby("order_date", as_index=False).agg(
    total_payment_value=("payment_value", "sum"),
    avg_payment_value=("payment_value", "mean"),
    avg_installments=("installments", "mean")
).rename(columns={"order_date": "Date"})

payments_daily.to_parquet(os.path.join(OUT_DIR, "payments_daily.parquet"), index=False)
payments_daily.head()

,Date,total_payment_value,avg_payment_value,avg_installments
0,2012-07-04,5123547.94,31626.839136,3.790123
1,2012-07-05,2751773.45,28368.798454,3.917526
2,2012-07-06,3054029.42,32839.026022,4.139785
3,2012-07-07,2667930.94,36546.999178,2.904110
4,2012-07-08,2360851.90,26827.862500,3.465909


In [6]:
# 5) order_items_daily (join orders -> order_date)
oi = read_tbl("order_items").copy()
oi_join = oi.merge(orders[["order_id", "order_date"]], on="order_id", how="left")
oi_join["order_date"] = to_dt(oi_join["order_date"])

oi_join["promo_line"] = oi_join["promo_id"].notna().astype(int)
oi_join["stacked_promo_line"] = (oi_join["promo_id"].notna() & oi_join["promo_id_2"].notna()).astype(int)

order_items_daily = oi_join.groupby("order_date", as_index=False).agg(
    total_quantity_sold=("quantity", "sum"),
    total_discount_amount=("discount_amount", "sum"),
    promo_line_count=("promo_line", "sum"),
    stacked_promo_count=("stacked_promo_line", "sum")
).rename(columns={"order_date": "Date"})

order_items_daily.to_parquet(os.path.join(OUT_DIR, "order_items_daily.parquet"), index=False)
order_items_daily.head()

,Date,total_quantity_sold,total_discount_amount,promo_line_count,stacked_promo_count
0,2012-07-04,777,0.0,0,0
1,2012-07-05,428,0.0,0,0
2,2012-07-06,441,0.0,0,0
3,2012-07-07,364,0.0,0,0
4,2012-07-08,394,0.0,0,0


In [7]:
# 6) promotions_daily (expand date range active)
promos = read_tbl("promotions").copy()
promos["start_date"] = to_dt(promos["start_date"])
promos["end_date"] = to_dt(promos["end_date"])

rows = []
for _, r in promos.dropna(subset=["start_date", "end_date"]).iterrows():
    rng = pd.date_range(r["start_date"], r["end_date"], freq="D")
    rows.append(pd.DataFrame({
        "Date": rng,
        "promo_id": r.get("promo_id", np.nan),
        "discount_value": r.get("discount_value", np.nan)
    }))

promo_expanded = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["Date", "promo_id", "discount_value"])

promotions_daily = promo_expanded.groupby("Date", as_index=False).agg(
    active_promo_count=("promo_id", "nunique"),
    avg_discount_value=("discount_value", "mean")
)
promotions_daily["has_any_promo"] = (promotions_daily["active_promo_count"] > 0).astype(int)

promotions_daily.to_parquet(os.path.join(OUT_DIR, "promotions_daily.parquet"), index=False)
promotions_daily.head()

,Date,active_promo_count,avg_discount_value,has_any_promo
0,2013-01-31,1,15.0,1
1,2013-02-01,1,15.0,1
2,2013-02-02,1,15.0,1
3,2013-02-03,1,15.0,1
4,2013-02-04,1,15.0,1


In [8]:
# 7) returns_daily
rets = read_tbl("returns").copy()
rets["return_date"] = to_dt(rets["return_date"])

return_id_col = "return_id" if "return_id" in rets.columns else rets.columns[0]
refund_col = "refund_amount" if "refund_amount" in rets.columns else None
return_agg = {"return_count": (return_id_col, "count")}
if refund_col is not None:
    return_agg["total_refund_amount"] = (refund_col, "sum")
returns_daily = rets.groupby("return_date", as_index=False).agg(**return_agg).rename(columns={"return_date": "Date"})
returns_daily.to_parquet(os.path.join(OUT_DIR, "returns_daily.parquet"), index=False)

# 8) reviews_daily
reviews = read_tbl("reviews").copy()
reviews["review_date"] = to_dt(reviews["review_date"])

review_id_col = "review_id" if "review_id" in reviews.columns else reviews.columns[0]
rating_col = "rating" if "rating" in reviews.columns else None
review_agg = {"review_count": (review_id_col, "count")}
if rating_col is not None:
    review_agg["avg_rating"] = (rating_col, "mean")
reviews_daily = reviews.groupby("review_date", as_index=False).agg(**review_agg).rename(columns={"review_date": "Date"})
reviews_daily.to_parquet(os.path.join(OUT_DIR, "reviews_daily.parquet"), index=False)

# 9) shipments_daily
ship = read_tbl("shipments").copy()
ship["ship_date"] = to_dt(ship["ship_date"])

if "delivery_date" in ship.columns:
    ship["delivery_date"] = to_dt(ship["delivery_date"])

    ship["delivery_days"] = (ship["delivery_date"] - ship["ship_date"]).dt.days
else:
    ship["delivery_days"] = np.nan

shipment_id_col = "shipment_id" if "shipment_id" in ship.columns else ("order_id" if "order_id" in ship.columns else ship.columns[0])
shipping_fee_col = "shipping_fee" if "shipping_fee" in ship.columns else None

ship_agg = {
    "shipment_count": (shipment_id_col, "count"),
    "avg_delivery_days": ("delivery_days", "mean")
}
if shipping_fee_col is not None:
    ship_agg["avg_shipping_fee"] = (shipping_fee_col, "mean")

shipments_daily = ship.groupby("ship_date", as_index=False).agg(**ship_agg).rename(columns={"ship_date": "Date"})
shipments_daily.to_parquet(os.path.join(OUT_DIR, "shipments_daily.parquet"), index=False)

# 10) inventory_monthly_summary
inv = read_tbl("inventory").copy()
inv["snapshot_date"] = to_dt(inv["snapshot_date"])

inv["year"] = inv["snapshot_date"].dt.year
inv["month"] = inv["snapshot_date"].dt.month
inventory_monthly_summary = inv.groupby(["year", "month"], as_index=False).agg(
    avg_stockout_days=("stockout_days", "mean"),
    avg_fill_rate=("fill_rate", "mean"),
    avg_days_of_supply=("days_of_supply", "mean"),
    avg_sell_through_rate=("sell_through_rate", "mean"),
    stockout_rate=("stockout_flag", "mean"),
    overstock_rate=("overstock_flag", "mean"),
    reorder_rate=("reorder_flag", "mean")
)
inventory_monthly_summary.to_parquet(os.path.join(OUT_DIR, "inventory_monthly_summary.parquet"), index=False)

print("Done building daily/monthly tables in", OUT_DIR)

Done building daily/monthly tables in D:\UNIVERSITY\PERSONAL_PROJECT\datathon\final\dataset\interim\daily
